# M13b · Optimisation sous contraintes et SVM — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookMaths/blob/main/chapitres/m13b_lagrange_svm/03_notebook.ipynb)

La partie 0 vérifie tes exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wbm.check`). « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/m13b_lagrange_svm/03_notebook.ipynb`, créée par `python tools/start_chapter.py M13b`) ; la version du dépôt est mise à jour par Claude. Sur Colab, le badge ouvre la version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | M13b.1 à M13b.13, M13b.20 | Les exercices papier (vérification) | ✏️ 🔗 | | |
| A | M13b.25 | Un problème quadratique sous contraintes d'égalité, par le système de Lagrange · `lagrange_equality_qp` | 🔨 | ★ | 20 |
|  | M13b.26 | La hinge loss et un sous-gradient de l'objectif d'une SVM linéaire · `hinge_loss, svm_subgradient` | 🔨 | ★★ | 25 |
|  | M13b.27 | Les noyaux RBF et polynomial · `rbf_kernel, polynomial_kernel` | 🔨 | ★ | 20 |
|  | M13b.28 | `scipy.optimize.minimize` sous contraintes et les noyaux de scikit-learn contre tes fonctions | 📦 | ★ | 15 |
| B | M13b.29 | Prédire : comment change le minimum quand on relâche la contrainte, et ce que dit λ | 🔮 | ★ | 10 |
|  | M13b.30 | La descente de sous-gradient sur la hinge loss : pas constant contre pas décroissant | 🔬 | ★★ | 20 |
|  | M13b.31 | Lignes de niveau et contrainte : voir le point de Lagrange | 🎨 | ★ | 20 |
| C | M13b.32 | Chasse à l'erreur : le signe d'un multiplicateur, une largeur de bande 1/‖w‖ au lieu de 2/‖w‖, une matrice de Gram non symétrique | 🐛 | ★ | 15 |
|  | M13b.33 | Vérifier une solution KKT : faisabilité, stationnarité, complémentarité | 🛠️ | ★ | 15 |
|  | M13b.34 | Défi : une SVM linéaire entraînée par sous-gradient, un objectif à 1 % de celui de scikit-learn au même C | 🏆 | ★★★ | 30 |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (a few seconds per cell); False = bigger experiments
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookMaths.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookMaths")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WBM_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wbm").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wbm").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier de workbookMaths.")

os.environ["WBM_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wbm  # noqa: E402

cfg = wbm.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M13b")  # your own library

## Objectifs et rappel express

- Vérifier tes calculs papier (partie 0) : Lagrange sous une et deux contraintes d'égalité, la distance à un plan, l'ACP relue par Lagrange, les conditions KKT, une fonction duale, la convexité, une SVM à marge dure, la hinge loss et le sous-gradient, le dual d'une SVM, les noyaux polynomial et RBF, la décision d'une SVM à noyau.
- Écrire les cinq fonctions de `mymath/svm.py` et les tester, puis comparer avec SciPy et scikit-learn.
- Prédire le prix d'une contrainte, comparer des pas de sous-gradient, voir le point de Lagrange, chasser des erreurs de SVM, vérifier une solution KKT, relever un défi d'entraînement.

**Rappel express.** Lagrange : $\nabla f + \lambda\nabla g = \mathbf{0}$ et $g = 0$, et $\frac{df^*}{dc} = -\lambda$ pour $f + \lambda(g - c)$ ; problème quadratique : $\begin{pmatrix} \mathbf{Q} & \mathbf{A}^\top \\ \mathbf{A} & \mathbf{0} \end{pmatrix}\begin{pmatrix} \mathbf{x} \\ \boldsymbol{\lambda} \end{pmatrix} = \begin{pmatrix} -\mathbf{c} \\ \mathbf{b} \end{pmatrix}$ ; KKT ($g_i \le 0$) : stationnarité, faisabilité, $\mu_i \ge 0$, $\mu_i g_i = 0$ ; dual $D(\boldsymbol{\mu}) = \min_{\mathbf{x}} \mathcal{L} \le f^*$ ; SVM : $\min \frac{1}{2}\lVert\mathbf{w}\rVert^2$ sous $y_i(\mathbf{w} \cdot \mathbf{x}_i + b) \ge 1$, marge $\rho = 1/\lVert\mathbf{w}\rVert$, bande $2\rho$ ; $J = \frac{1}{2}\lVert\mathbf{w}\rVert^2 + C\sum_i \max(0, 1 - t_i)$, $\mathbf{g}_{\mathbf{w}} = \mathbf{w} - C\sum_{t_i < 1} y_i\mathbf{x}_i$, $g_b = -C\sum_{t_i < 1} y_i$ ; dual : $\mathbf{w} = \sum_i \alpha_i y_i \mathbf{x}_i$, $0 \le \alpha_i \le C$ ; noyaux $(\mathbf{x} \cdot \mathbf{z} + c)^d$ et $e^{-\gamma\lVert\mathbf{x} - \mathbf{z}\rVert^2}$, décision $s(\mathbf{x}) = \sum_i \alpha_i y_i\,k(\mathbf{x}_i, \mathbf{x}) + b$.

## Partie 0 · Vérifier tes exercices papier (✏️ M13b.1 à M13b.13, 🔗 M13b.20)

Fais d'abord les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse ici : **la valeur** que tu as trouvée, sous la forme indiquée en commentaire, jamais le calcul qui y mène :
- un entier, ou un nombre avec un point décimal, `-1.25`, arrondi comme le demande la question ;
- une fraction irréductible entre guillemets, `"5/9"` ;
- une liste `[0.15, -2]` ; dans une liste, une fraction s'écrit comme un calcul, sans guillemets : `[2/9, 7/9]` ; une matrice, comme une liste de lignes : `[[1, 0.5], [0.5, 1]]` ;
- une expression entre guillemets, écrite comme en Python : `"mu**2 - 7*mu"` pour $\mu^2 - 7\mu$ ;
- des lettres collées entre guillemets, `"BD"`, ou `True` / `False`.

Les réponses pas encore remplies affichent ⏳.

In [ ]:
from fractions import Fraction   # Fraction(2, 9) is accepted too, as "2/9"

**Ex M13b.1 — Lagrange à la main : un produit maximal sous une contrainte linéaire**

In [ ]:
# Ex M13b.1: the values you found on paper
answer_M13b_1a = ...  # a) a list of 3 numbers
answer_M13b_1b = ...  # b) a number
answer_M13b_1c = ...  # c) a number
answer_M13b_1d = ...  # d) a number
answer_M13b_1e = ...  # e) an irreducible fraction, as text
answer_M13b_1f = ...  # f) a number
answer_M13b_1g = ...  # g) a number

for letter, answer in zip("abcdefg", [answer_M13b_1a, answer_M13b_1b, answer_M13b_1c, answer_M13b_1d, answer_M13b_1e, answer_M13b_1f, answer_M13b_1g]):
    wbm.check(f"M13b.1{letter}", answer)

**Ex M13b.2 — La distance d'un point à un plan, retrouvée par Lagrange**

In [ ]:
# Ex M13b.2: the values you found on paper
answer_M13b_2a = ...  # a) an irreducible fraction, as text
answer_M13b_2b = ...  # b) a list of 3 fractions, written as calculations or with 6 decimals
answer_M13b_2c = ...  # c) a number
answer_M13b_2d = ...  # d) a number
answer_M13b_2e = ...  # e) a number
answer_M13b_2f = ...  # f) a number

for letter, answer in zip("abcdef", [answer_M13b_2a, answer_M13b_2b, answer_M13b_2c, answer_M13b_2d, answer_M13b_2e, answer_M13b_2f]):
    wbm.check(f"M13b.2{letter}", answer)

**Ex M13b.3 — Un problème quadratique sous deux contraintes linéaires : le système de Lagrange**

In [ ]:
# Ex M13b.3: the values you found on paper
answer_M13b_3a = ...  # a) an integer
answer_M13b_3b = ...  # b) a list of 3 fractions, written as calculations or with 6 decimals
answer_M13b_3c = ...  # c) a list of 2 fractions, written as calculations or with 6 decimals
answer_M13b_3d = ...  # d) an irreducible fraction, as text
answer_M13b_3e = ...  # e) an integer
answer_M13b_3f = ...  # f) a list of 3 numbers
answer_M13b_3g = ...  # g) a number
answer_M13b_3h = ...  # h) an irreducible fraction, as text

for letter, answer in zip("abcdefgh", [answer_M13b_3a, answer_M13b_3b, answer_M13b_3c, answer_M13b_3d, answer_M13b_3e, answer_M13b_3f, answer_M13b_3g, answer_M13b_3h]):
    wbm.check(f"M13b.3{letter}", answer)

**Ex M13b.4 — Relire l'ACP par Lagrange : maximiser uᵀCu sous ‖u‖ = 1**

In [ ]:
# Ex M13b.4: the values you found on paper
answer_M13b_4a = ...  # a) an integer
answer_M13b_4b = ...  # b) a list of 3 numbers (4 decimals)
answer_M13b_4c = ...  # c) a number
answer_M13b_4d = ...  # d) a number
answer_M13b_4e = ...  # e) an integer
answer_M13b_4f = ...  # f) an irreducible fraction, as text
answer_M13b_4g = ...  # g) an irreducible fraction, as text
answer_M13b_4h = ...  # h) a number

for letter, answer in zip("abcdefgh", [answer_M13b_4a, answer_M13b_4b, answer_M13b_4c, answer_M13b_4d, answer_M13b_4e, answer_M13b_4f, answer_M13b_4g, answer_M13b_4h]):
    wbm.check(f"M13b.4{letter}", answer)

**Ex M13b.5 — Une contrainte d'inégalité : active ou non, les conditions KKT à la main**

In [ ]:
# Ex M13b.5: the values you found on paper
answer_M13b_5a = ...  # a) True or False
answer_M13b_5b = ...  # b) an irreducible fraction, as text
answer_M13b_5c = ...  # c) a list of 2 fractions, written as calculations or with 6 decimals
answer_M13b_5d = ...  # d) a number
answer_M13b_5e = ...  # e) a number
answer_M13b_5f = ...  # f) an irreducible fraction, as text
answer_M13b_5g = ...  # g) an irreducible fraction, as text

for letter, answer in zip("abcdefg", [answer_M13b_5a, answer_M13b_5b, answer_M13b_5c, answer_M13b_5d, answer_M13b_5e, answer_M13b_5f, answer_M13b_5g]):
    wbm.check(f"M13b.5{letter}", answer)

**Ex M13b.6 — La fonction duale d'un petit problème quadratique, et la dualité faible**

In [ ]:
# Ex M13b.6: the values you found on paper
answer_M13b_6a = ...  # a) an expression in mu (between quotes, "mu**3" for μ³)
answer_M13b_6b = ...  # b) a number
answer_M13b_6c = ...  # c) a number
answer_M13b_6d = ...  # d) a number
answer_M13b_6e = ...  # e) a list of 2 numbers
answer_M13b_6f = ...  # f) a number
answer_M13b_6g = ...  # g) a number

for letter, answer in zip("abcdefg", [answer_M13b_6a, answer_M13b_6b, answer_M13b_6c, answer_M13b_6d, answer_M13b_6e, answer_M13b_6f, answer_M13b_6g]):
    wbm.check(f"M13b.6{letter}", answer)

**Ex M13b.7 — Reconnaître un problème convexe : fonctions, contraintes, ensembles**

In [ ]:
# Ex M13b.7: the values you found on paper
answer_M13b_7a = ...  # a) letters, as text

for letter, answer in zip("a", [answer_M13b_7a]):
    wbm.check(f"M13b.7{letter}", answer)

**Ex M13b.8 — Une SVM à marge dure sur trois points, résolue par les conditions KKT**

In [ ]:
# Ex M13b.8: the values you found on paper
answer_M13b_8a = ...  # a) a list of 2 numbers
answer_M13b_8b = ...  # b) a number
answer_M13b_8c = ...  # c) a list of 3 numbers
answer_M13b_8d = ...  # d) a number
answer_M13b_8e = ...  # e) a number (4 decimals)
answer_M13b_8f = ...  # f) a number
answer_M13b_8g = ...  # g) an integer
answer_M13b_8h = ...  # h) a number

for letter, answer in zip("abcdefgh", [answer_M13b_8a, answer_M13b_8b, answer_M13b_8c, answer_M13b_8d, answer_M13b_8e, answer_M13b_8f, answer_M13b_8g, answer_M13b_8h]):
    wbm.check(f"M13b.8{letter}", answer)

**Ex M13b.9 — La hinge loss et l'objectif régularisé de quelques points**

In [ ]:
# Ex M13b.9: the values you found on paper
answer_M13b_9a = ...  # a) a list of 6 numbers
answer_M13b_9b = ...  # b) a list of 6 numbers
answer_M13b_9c = ...  # c) a number (4 decimals)
answer_M13b_9d = ...  # d) a number
answer_M13b_9e = ...  # e) a number
answer_M13b_9f = ...  # f) an irreducible fraction, as text
answer_M13b_9g = ...  # g) an integer
answer_M13b_9h = ...  # h) an integer

for letter, answer in zip("abcdefgh", [answer_M13b_9a, answer_M13b_9b, answer_M13b_9c, answer_M13b_9d, answer_M13b_9e, answer_M13b_9f, answer_M13b_9g, answer_M13b_9h]):
    wbm.check(f"M13b.9{letter}", answer)

**Ex M13b.10 — Un pas de sous-gradient sur l'objectif d'une SVM linéaire**

In [ ]:
# Ex M13b.10: the values you found on paper
answer_M13b_10a = ...  # a) a list of 4 numbers
answer_M13b_10b = ...  # b) a number (4 decimals)
answer_M13b_10c = ...  # c) a list of 2 numbers
answer_M13b_10d = ...  # d) a number
answer_M13b_10e = ...  # e) a list of 2 numbers
answer_M13b_10f = ...  # f) a number (4 decimals)
answer_M13b_10g = ...  # g) a list of 2 numbers

for letter, answer in zip("abcdefg", [answer_M13b_10a, answer_M13b_10b, answer_M13b_10c, answer_M13b_10d, answer_M13b_10e, answer_M13b_10f, answer_M13b_10g]):
    wbm.check(f"M13b.10{letter}", answer)

**Ex M13b.11 — Le dual d'une SVM : lire les vecteurs de support et retrouver w**

In [ ]:
# Ex M13b.11: the values you found on paper
answer_M13b_11a = ...  # a) a number
answer_M13b_11b = ...  # b) a list of 2 numbers
answer_M13b_11c = ...  # c) a number
answer_M13b_11d = ...  # d) a list of integers
answer_M13b_11e = ...  # e) a list of 6 numbers
answer_M13b_11f = ...  # f) a number
answer_M13b_11g = ...  # g) a number

for letter, answer in zip("abcdefg", [answer_M13b_11a, answer_M13b_11b, answer_M13b_11c, answer_M13b_11d, answer_M13b_11e, answer_M13b_11f, answer_M13b_11g]):
    wbm.check(f"M13b.11{letter}", answer)

**Ex M13b.12 — Le noyau (1 + x · z)² en dimension 2 : sa fonction φ à six coordonnées**

In [ ]:
# Ex M13b.12: the values you found on paper
answer_M13b_12a = ...  # a) a list of 6 numbers (2 decimals)
answer_M13b_12b = ...  # b) a list of 6 numbers (2 decimals)
answer_M13b_12c = ...  # c) an integer
answer_M13b_12d = ...  # d) a list of 6 numbers (2 decimals)
answer_M13b_12e = ...  # e) an integer
answer_M13b_12f = ...  # f) an integer

for letter, answer in zip("abcdef", [answer_M13b_12a, answer_M13b_12b, answer_M13b_12c, answer_M13b_12d, answer_M13b_12e, answer_M13b_12f]):
    wbm.check(f"M13b.12{letter}", answer)

**Ex M13b.13 — La matrice de Gram d'un noyau RBF sur trois points, et l'effet de γ**

In [ ]:
# Ex M13b.13: the values you found on paper
answer_M13b_13a = ...  # a) a list of 3 numbers
answer_M13b_13b = ...  # b) a number (4 decimals)
answer_M13b_13c = ...  # c) a 3 x 3 matrix, as a list of rows (4 decimals)
answer_M13b_13d = ...  # d) a number (4 decimals)
answer_M13b_13e = ...  # e) a number (4 decimals)
answer_M13b_13f = ...  # f) a number
answer_M13b_13g = ...  # g) an integer
answer_M13b_13h = ...  # h) an integer

for letter, answer in zip("abcdefgh", [answer_M13b_13a, answer_M13b_13b, answer_M13b_13c, answer_M13b_13d, answer_M13b_13e, answer_M13b_13f, answer_M13b_13g, answer_M13b_13h]):
    wbm.check(f"M13b.13{letter}", answer)

**Ex M13b.20 — Pont : la décision d'une SVM à noyau, une somme pondérée sur les vecteurs de support**

In [ ]:
# Ex M13b.20: the values you found on paper
answer_M13b_20a = ...  # a) a number
answer_M13b_20b = ...  # b) a number (4 decimals)
answer_M13b_20c = ...  # c) an integer (-1 or 1)
answer_M13b_20d = ...  # d) a number (4 decimals)
answer_M13b_20e = ...  # e) a number
answer_M13b_20f = ...  # f) an integer

for letter, answer in zip("abcdef", [answer_M13b_20a, answer_M13b_20b, answer_M13b_20c, answer_M13b_20d, answer_M13b_20e, answer_M13b_20f]):
    wbm.check(f"M13b.20{letter}", answer)

## Partie A · Ta librairie : Lagrange, hinge loss, sous-gradient et noyaux ; SciPy et scikit-learn

Fiche §2, §7 et §8. Les cinq fonctions de `mymath/svm.py`, puis `scipy.optimize.minimize` et les noyaux et la hinge loss de scikit-learn.

In [ ]:
# Tools of the notebook (parts A to C)
import inspect
import math
import re
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np

from wbm.nbtools import checked, drawn, fr, run_pytest_group, verdict


def raised(function, *args, **kwargs):
    """The name of the exception that function(*args, **kwargs) raises, or "aucune" if it returns.

    NotImplementedError (a TODO not done yet) goes through, for wbm.attempt to show ⏳; so do the errors of a code
    that cannot run (a missing import, a misspelt name such as ValueErorr), for the check to name them."""
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)
            function(*args, **kwargs)
    except (NotImplementedError, NameError, AttributeError, ImportError):
        raise
    except Exception as exc:  # noqa: BLE001 - any other error is the answer
        return type(exc).__name__
    return "aucune"

### Ex M13b.25 — Un problème quadratique sous contraintes d'égalité, par le système de Lagrange · `lagrange_equality_qp` 🔨 ★ ⏱️ 20 min
**Objectif :** programmer la résolution d'un problème quadratique sous contraintes d'égalité par un seul système linéaire par blocs, et vérifier que sa solution est bien le minimum.  
**Prérequis :** fiche §2 · **mymath :** `svm.py` · **Parcours :** C, E

> **Mode d'emploi `mymath`** (comme en M03). Ouvre ta copie `mon_travail/mymath/svm.py`, lis la docstring du module puis celle de la fonction, remplace la ligne `raise NotImplementedError(...)` par ton code, **enregistre** le fichier, puis exécute les cellules ci-dessous : la première recharge ta librairie. NumPy est permis (`@`, `.T`, `np.linalg.norm` et `np.linalg.solve` compris) ; `scipy.optimize`, `scipy.spatial`, `scipy.linalg` et scikit-learn sont interdits (les tests comparent tes résultats aux leurs, et un test vérifie que tu ne t'en sers pas). Ne modifie jamais les tableaux reçus : travaille sur une copie, `np.array(X, dtype=float)`. Les nombres se rendent en `float` de Python, les vecteurs et les matrices en tableaux NumPy de flottants ; les labels d'une SVM valent −1 ou +1, un exemple par ligne.

Écris `lagrange_equality_qp(Q, c, A, b)` (fiche §2), en suivant sa docstring : les contrôles de forme d'abord (`ValueError` si `Q` n'est pas $n \times n$ pour un `c` de longueur $n$, si `A` n'a pas $n$ colonnes et une ligne par second membre de `b`) ; puis la matrice par blocs $\begin{pmatrix} \mathbf{Q} & \mathbf{A}^\top \\ \mathbf{A} & \mathbf{0} \end{pmatrix}$ (`np.block`), le second membre $(-\mathbf{c}, \mathbf{b})$ (`np.concatenate`), `np.linalg.solve`, et le découpage de la solution en $(\mathbf{x}, \boldsymbol{\lambda})$. Si le système est singulier, `np.linalg.solve` lève une `numpy.linalg.LinAlgError` : attrape-la et lève une `ValueError` à la place.

La vérification calcule, avec le problème de la cellule fournie (4 variables, 2 contraintes) :
- a) le point $\mathbf{x}$ rendu par `lagrange_equality_qp(Q_25, C_25, A_25, B_25)` (une liste, 6 décimales) ;
- b) les multiplicateurs $\boldsymbol{\lambda}$ (une liste, 6 décimales) ;
- c) vrai ou faux : $\mathbf{Q}\mathbf{x} + \mathbf{c} + \mathbf{A}^\top\boldsymbol{\lambda} = \mathbf{0}$, chaque coordonnée à $10^{-10}$ près ;
- d) vrai ou faux : $\mathbf{x}$ est admissible ($\mathbf{A}\mathbf{x} = \mathbf{b}$ à $10^{-10}$ près) et aucun des 1 000 points admissibles tirés au hasard (graine 4225) n'a une valeur de $f$ plus petite ;
- e) le nom de l'erreur levée quand la même contrainte est écrite deux fois, `lagrange_equality_qp(Q_25, C_25, TWICE_25, [5, 5])`.

Dans ta copie : écris le système par blocs de ce problème (sa taille ?). Que vaut $f(\mathbf{x})$, et pourquoi d) suffit-il à te convaincre que c'est un minimum, alors que la condition de Lagrange n'est que nécessaire (fiche §5) ? Pourquoi deux contraintes identiques rendent-elles la matrice par blocs singulière (regarde ses deux dernières lignes) ? Quel multiplicateur dit que la première contrainte coûte le plus cher ?

In [ ]:
Q_25 = np.array([[6.0, 1.0, 0.0, -1.0],
                 [1.0, 4.0, 1.0, 0.0],
                 [0.0, 1.0, 5.0, 2.0],
                 [-1.0, 0.0, 2.0, 3.0]])         # symmetric positive definite
C_25 = np.array([-2.0, 3.0, -1.0, 4.0])
A_25 = np.array([[1.0, -2.0, 1.0, 3.0],
                 [2.0, 0.0, -1.0, 1.0]])         # two independent constraints A x = b
B_25 = np.array([5.0, -3.0])
TWICE_25 = [[1.0, -2.0, 1.0, 3.0], [1.0, -2.0, 1.0, 3.0]]   # the same constraint written twice

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M13b")  # reload your saved file

In [ ]:
with checked("M13b.25a"):
    wbm.check("M13b.25a", np.asarray(mymath.svm.lagrange_equality_qp(Q_25, C_25, A_25, B_25)[0]).tolist(), computed=True)
with checked("M13b.25b"):
    wbm.check("M13b.25b", np.asarray(mymath.svm.lagrange_equality_qp(Q_25, C_25, A_25, B_25)[1]).tolist(), computed=True)
with checked("M13b.25c"):
    x_25, lam_25 = mymath.svm.lagrange_equality_qp(Q_25, C_25, A_25, B_25)
    wbm.check("M13b.25c", bool(np.max(np.abs(Q_25 @ x_25 + C_25 + A_25.T @ lam_25)) <= 1e-10), computed=True)
with checked("M13b.25d"):
    x_25 = np.asarray(mymath.svm.lagrange_equality_qp(Q_25, C_25, A_25, B_25)[0], dtype=float)
    base_25 = A_25.T @ np.linalg.solve(A_25 @ A_25.T, B_25)                    # one feasible point
    along_25 = np.eye(4) - A_25.T @ np.linalg.solve(A_25 @ A_25.T, A_25)      # the projection on A v = 0
    tries_25 = base_25 + np.random.default_rng(4225).normal(0.0, 1.0, size=(1000, 4)) @ along_25
    f_tries_25 = 0.5 * np.einsum("ij,jk,ik->i", tries_25, Q_25, tries_25) + tries_25 @ C_25
    wbm.check("M13b.25d", bool(np.max(np.abs(A_25 @ x_25 - B_25)) <= 1e-10
                               and np.all(f_tries_25 >= 0.5 * x_25 @ Q_25 @ x_25 + C_25 @ x_25 - 1e-12)), computed=True)
with checked("M13b.25e"):
    wbm.check("M13b.25e", raised(mymath.svm.lagrange_equality_qp, Q_25, C_25, TWICE_25, [5, 5]), computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_m13b_svm.py", "test_lagrange or test_svm_uses_")

### Ex M13b.26 — La hinge loss et un sous-gradient de l'objectif d'une SVM linéaire · `hinge_loss, svm_subgradient` 🔨 ★★ ⏱️ 25 min
**Objectif :** programmer la hinge loss et un sous-gradient de l'objectif d'une SVM à marge souple, avec la bonne convention au coude, et voir qu'un pas de sous-gradient peut faire monter l'objectif.  
**Prérequis :** fiche §7 · **mymath :** `svm.py` · **Parcours :** C, E

Écris, en suivant les docstrings (fiche §7) :
- `hinge_loss(scores, y)` : `ValueError` si les scores sont vides ou pas 1-D, ou si `y` n'a pas un label −1 ou +1 par score ; sinon la **moyenne** des $\max(0, 1 - y_is_i)$ (`np.maximum`) ;
- `svm_subgradient(w, b, X, y, C=1.0)` : `ValueError` si les formes ne vont pas ensemble, si un label n'est ni −1 ni +1, ou si $C \le 0$ ; sinon le triplet $(J, \mathbf{g}_{\mathbf{w}}, g_b)$, avec $J = \frac{1}{2}\lVert\mathbf{w}\rVert^2 + C\sum_i \max(0, 1 - t_i)$ (une **somme** de hinges) et le sous-gradient de la fiche, qui ne compte que les exemples de marge fonctionnelle $t_i = y_i(\mathbf{w} \cdot \mathbf{x}_i + b)$ **strictement** inférieure à 1 : $\mathbf{g}_{\mathbf{w}} = \mathbf{w} - C\sum_{t_i < 1} y_i\mathbf{x}_i$ et $g_b = -C\sum_{t_i < 1} y_i$.

Les huit exemples de la cellule fournie ont des coordonnées multiples de 0,5 : avec $(\mathbf{w}, b)$ donné, le premier a une marge fonctionnelle **exactement** 1, en flottants aussi. La vérification calcule :
- a) `hinge_loss(X_26 @ W_26 + B_26, Y_26)` (6 décimales) ;
- b) $J$, le premier élément de `svm_subgradient(W_26, B_26, X_26, Y_26, C=C_26)` (6 décimales) ;
- c) $\mathbf{g}_{\mathbf{w}}$ (une liste, 6 décimales) ;
- d) $g_b$ (6 décimales) ;
- e) vrai ou faux : l'inégalité du sous-gradient, $J(\mathbf{v}, d) \ge J(\mathbf{w}, b) + \mathbf{g}_{\mathbf{w}} \cdot (\mathbf{v} - \mathbf{w}) + g_b\,(d - b)$, tient (à $10^{-12}$ près) pour 500 points $(\mathbf{v}, d)$ tirés au hasard (graine 4226) ;
- f) $J$ après un pas de taille $\eta = 0{,}5$, $(\mathbf{w}, b) \leftarrow (\mathbf{w}, b) - 0{,}5\,(\mathbf{g}_{\mathbf{w}}, g_b)$ (4 décimales) ;
- g) le nom de l'erreur levée par `svm_subgradient(W_26, B_26, X_26, LABEL_0_26, C=C_26)`.

Dans ta copie : calcule à la main les huit marges fonctionnelles, et range les exemples selon leur marge fonctionnelle ($t_i > 1$, $0 \le t_i \le 1$, $t_i < 0$). Retrouve a), b) et c). Que deviendrait $\mathbf{g}_{\mathbf{w}}$ si l'on comptait aussi le premier exemple, et pourquoi est-ce aussi un sous-gradient ? Compare f) à b) : que dit la fiche de ce phénomène, et le pas $\eta = 0{,}1$ ferait-il de même ?

In [ ]:
X_26 = np.array([[2.0, 1.5], [3.0, -1.0], [0.5, 2.5], [-1.0, -0.5],
                 [1.5, 3.0], [-2.0, 1.0], [0.0, -2.0], [2.5, 0.5]])   # 8 examples (multiples of 0.5: exact in floats)
Y_26 = np.array([1, 1, 1, -1, -1, -1, 1, -1])                          # their labels
W_26, B_26, C_26 = np.array([0.75, -0.5]), 0.25, 0.5                   # a separator (w, b) and the weight C
LABEL_0_26 = [1, 1, 1, 0, -1, -1, 1, -1]                               # a label 0 by mistake

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M13b")  # reload your saved file

In [ ]:
with checked("M13b.26a"):
    wbm.check("M13b.26a", mymath.svm.hinge_loss(X_26 @ W_26 + B_26, Y_26), computed=True)
with checked("M13b.26b"):
    wbm.check("M13b.26b", mymath.svm.svm_subgradient(W_26, B_26, X_26, Y_26, C=C_26)[0], computed=True)
with checked("M13b.26c"):
    wbm.check("M13b.26c", np.asarray(mymath.svm.svm_subgradient(W_26, B_26, X_26, Y_26, C=C_26)[1]).tolist(), computed=True)
with checked("M13b.26d"):
    wbm.check("M13b.26d", mymath.svm.svm_subgradient(W_26, B_26, X_26, Y_26, C=C_26)[2], computed=True)
with checked("M13b.26e"):
    j_26, gw_26, gb_26 = mymath.svm.svm_subgradient(W_26, B_26, X_26, Y_26, C=C_26)
    rng_26 = np.random.default_rng(4226)
    v_26, d_26 = W_26 + rng_26.normal(0.0, 1.5, size=(500, 2)), B_26 + rng_26.normal(0.0, 1.5, size=500)
    wbm.check("M13b.26e", all(mymath.svm.svm_subgradient(v, float(d), X_26, Y_26, C=C_26)[0]
                              >= j_26 + float(np.dot(gw_26, v - W_26)) + gb_26 * (d - B_26) - 1e-12
                              for v, d in zip(v_26, d_26)), computed=True)
with checked("M13b.26f"):
    j_26, gw_26, gb_26 = mymath.svm.svm_subgradient(W_26, B_26, X_26, Y_26, C=C_26)
    wbm.check("M13b.26f", mymath.svm.svm_subgradient(W_26 - 0.5 * np.asarray(gw_26), B_26 - 0.5 * gb_26, X_26, Y_26,
                                                     C=C_26)[0], computed=True)
with checked("M13b.26g"):
    wbm.check("M13b.26g", raised(mymath.svm.svm_subgradient, W_26, B_26, X_26, LABEL_0_26, C=C_26), computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_m13b_svm.py", "test_hinge or test_subgradient or test_svm_uses_")

### Ex M13b.27 — Les noyaux RBF et polynomial · `rbf_kernel, polynomial_kernel` 🔨 ★ ⏱️ 20 min
**Objectif :** programmer les matrices des noyaux RBF et polynomial en un seul produit matriciel, sans erreur d'arrondi qui dépasse 1.  
**Prérequis :** fiche §8 · **mymath :** `svm.py` · **Parcours :** C, E

Écris, en suivant les docstrings (fiche §8) :
- `rbf_kernel(X, Z, gamma)` : `ValueError` si `X` ou `Z` n'est pas 2-D, si leurs nombres de colonnes diffèrent, ou si $\gamma \le 0$ ; sinon la matrice $\mathbf{K}_{ij} = e^{-\gamma\lVert\mathbf{x}_i - \mathbf{z}_j\rVert^2}$, ligne $i$ pour $\mathbf{x}_i$, colonne $j$ pour $\mathbf{z}_j$. Calcule tous les carrés de distances d'un coup en développant $\lVert\mathbf{x} - \mathbf{z}\rVert^2 = \lVert\mathbf{x}\rVert^2 - 2\,\mathbf{x} \cdot \mathbf{z} + \lVert\mathbf{z}\rVert^2$ (un produit `X @ Z.T` et deux sommes de carrés), puis remplace par 0 les petites valeurs négatives que l'arrondi peut produire (`np.maximum`) ;
- `polynomial_kernel(X, Z, degree=2, coef0=1.0)` : les mêmes contrôles de forme, `ValueError` si `degree` n'est pas un entier $\ge 1$ ou si `coef0` est négatif ; sinon $\mathbf{K}_{ij} = (\mathbf{x}_i \cdot \mathbf{z}_j + c)^d$.

La vérification calcule, avec les points de la cellule fournie :
- a) `rbf_kernel(X_27, Z_27, 0.4)` (une matrice $3 \times 2$, comme une liste de lignes, 6 décimales) ;
- b) `polynomial_kernel(X_27, Z_27, degree=3, coef0=0.5)` (6 décimales) ;
- c) vrai ou faux : la matrice de Gram `rbf_kernel(G_27, G_27, 0.7)` des 30 points est symétrique (à $10^{-12}$ près) et ses valeurs propres sont toutes $\ge -10^{-10}$ ;
- d) vrai ou faux : toutes les valeurs de `rbf_kernel(F_27, F_27, 0.5)` sont $\le 1$, pour 40 points de coordonnées voisines de 3 000 ;
- e) le nom de l'erreur levée par `rbf_kernel(X_27, Z_27, 0.0)`.

Dans ta copie : retrouve à la main une valeur de a) (les carrés des distances d'abord) et une de b). Pourquoi la matrice de c) n'est-elle pas **exactement** symétrique en flottants, alors qu'elle l'est en théorie ? Pour d), que vaudrait la diagonale sans `np.maximum`, et pourquoi le développement du carré perd-il des chiffres quand les points sont loin de l'origine (compare $\lVert\mathbf{x}\rVert^2 \approx 2{,}7 \times 10^7$ à l'écart cherché) ? Pourquoi $\gamma = 0$ est-il refusé ?

In [ ]:
X_27 = np.array([[0.5, -1.0], [1.5, 0.0], [-0.5, 1.0]])    # 3 points x_i
Z_27 = np.array([[1.0, 0.5], [-1.5, -0.5]])               # 2 points z_j
rng_27 = np.random.default_rng(4227)
G_27 = rng_27.normal(0.0, 1.0, size=(30, 3))               # 30 points, for a Gram matrix
F_27 = 3000.0 + rng_27.normal(0.0, 1.0, size=(40, 3))     # 40 points far from the origin, close to one another

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M13b")  # reload your saved file

In [ ]:
with checked("M13b.27a"):
    wbm.check("M13b.27a", np.asarray(mymath.svm.rbf_kernel(X_27, Z_27, 0.4)).tolist(), computed=True)
with checked("M13b.27b"):
    wbm.check("M13b.27b", np.asarray(mymath.svm.polynomial_kernel(X_27, Z_27, degree=3, coef0=0.5)).tolist(),
              computed=True)
with checked("M13b.27c"):
    k_27 = np.asarray(mymath.svm.rbf_kernel(G_27, G_27, 0.7), dtype=float)
    wbm.check("M13b.27c", bool(k_27.shape == (30, 30) and np.allclose(k_27, k_27.T, rtol=0, atol=1e-12)
                               and np.linalg.eigvalsh((k_27 + k_27.T) / 2).min() >= -1e-10), computed=True)
with checked("M13b.27d"):
    wbm.check("M13b.27d", bool(np.max(mymath.svm.rbf_kernel(F_27, F_27, 0.5)) <= 1.0), computed=True)
with checked("M13b.27e"):
    wbm.check("M13b.27e", raised(mymath.svm.rbf_kernel, X_27, Z_27, 0.0), computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_m13b_svm.py", "test_rbf or test_polynomial or test_svm_uses_")

### Ex M13b.28 — `scipy.optimize.minimize` sous contraintes et les noyaux de scikit-learn contre tes fonctions 📦 ★ ⏱️ 15 min
**Objectif :** résoudre un problème sous contraintes avec `scipy.optimize.minimize`, utiliser les noyaux et la hinge loss de scikit-learn, lire leurs paramètres par défaut, et comparer avec ta librairie.  
**Prérequis :** fiche §2 et §8 ; Ex M13b.25 à M13b.27 · **Parcours :** C, E

En pratique, ces calculs sont tout faits :
- `minimize(fun, x0, jac=grad, method="SLSQP", constraints=[{"type": "eq", "fun": h, "jac": dh}], tol=1e-12)` (SciPy) part de `x0` et cherche le minimum de `fun` sous la contrainte $h(\mathbf{x}) = \mathbf{0}$ (`h` rend un vecteur, une coordonnée par contrainte ; `grad` et `dh` rendent le gradient de `fun` et la jacobienne de `h`) ; elle rend un objet `res` : `res.x` est le point trouvé, `res.fun` la valeur de `fun` en ce point, `res.nit` le nombre d'itérations ;
- `rbf_kernel(X, Z, gamma=g)` (scikit-learn) rend la matrice des $e^{-g\lVert\mathbf{x}_i - \mathbf{z}_j\rVert^2}$ ; sans `gamma`, elle prend $g = 1/p$, avec $p$ le nombre de features ;
- `polynomial_kernel(X, Z, degree=d, gamma=g, coef0=r)` rend la matrice des $(g\,\mathbf{x}_i \cdot \mathbf{z}_j + r)^d$ ; sans paramètres, $d = 3$, $g = 1/p$ et $r = 1$ ;
- `hinge_loss(y, s)` rend la hinge **moyenne** des scores `s` pour les labels `y` : les labels **d'abord**, à l'inverse de ta fonction.

Calcule :
- a) `res.fun` pour le problème de M13b.25 (`Q_25`, `C_25`, `A_25`, `B_25`), avec `fun` $= \frac{1}{2}\mathbf{x}^\top\mathbf{Q}\mathbf{x} + \mathbf{c}^\top\mathbf{x}$, son gradient, la contrainte $\mathbf{A}\mathbf{x} - \mathbf{b} = \mathbf{0}$ et sa jacobienne $\mathbf{A}$, depuis `x0 = np.zeros(4)`, avec `tol=1e-12` (6 décimales) ;
- b) vrai ou faux : `res.x` et le $\mathbf{x}$ de ta fonction `lagrange_equality_qp` sont égaux à $10^{-6}$ près (`np.allclose(..., rtol=0, atol=1e-6)`) ;
- c) vrai ou faux : `rbf_kernel(X_28, Z_28, gamma=0.3)` de scikit-learn et ta fonction `rbf_kernel(X_28, Z_28, 0.3)` sont égales à $10^{-12}$ près ;
- d) vrai ou faux : `polynomial_kernel(X_28, Z_28, degree=3, gamma=1.0, coef0=0.5)` et ta fonction `polynomial_kernel(X_28, Z_28, degree=3, coef0=0.5)` sont égales à $10^{-12}$ près (relativement : `np.allclose(..., rtol=1e-12, atol=1e-12)`) ;
- e) la valeur `[0, 0]` de `polynomial_kernel(P_28, R_28)` de scikit-learn, **sans aucun paramètre** (6 décimales). Ne devine pas : **exécute-la** ;
- f) `hinge_loss(Y_28, S_28)` de scikit-learn (6 décimales) ;
- g) vrai ou faux : ta fonction `hinge_loss(S_28, Y_28)` est égale à f), à $10^{-12}$ près ;
- h) le nom de l'erreur levée par `hinge_loss(S_28, Y_28)` de scikit-learn, les arguments dans l'ordre de ta fonction (exécute-la dans un `try` / `except Exception as exc`, puis `type(exc).__name__`).

Dans ta copie : combien d'itérations SLSQP a-t-elle faites (`res.nit`), et pourquoi ta fonction n'en a-t-elle besoin d'aucune ? Retrouve e) à la main à partir de `P_28[0] @ R_28[0]` : quels sont les trois paramètres par défaut, et quelle valeur aurait donnée ta fonction avec ses propres défauts ? Que penses-tu d'un message d'erreur comme celui de h) quand on inverse deux arguments ?

In [ ]:
from scipy.optimize import minimize
from sklearn.metrics import hinge_loss
from sklearn.metrics.pairwise import polynomial_kernel, rbf_kernel

rng_28 = np.random.default_rng(4228)
X_28 = np.round(rng_28.normal(0.0, 1.0, size=(5, 3)), 2)     # 5 points
Z_28 = np.round(rng_28.normal(0.0, 1.0, size=(4, 3)), 2)     # 4 other points
P_28 = np.round(rng_28.normal(0.0, 1.0, size=(3, 4)), 2)     # 3 points with p = 4 features
R_28 = np.round(rng_28.normal(0.0, 1.0, size=(2, 4)), 2)     # 2 other points with 4 features
Y_28 = np.where(rng_28.random(12) < 0.5, -1, 1)              # 12 labels
S_28 = np.round(rng_28.normal(0.3, 1.5, size=12), 2)         # their scores

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M13b")
answer_M13b_28a = ...  # a) a number (6 decimals)
answer_M13b_28e = ...  # e) a number (6 decimals)
answer_M13b_28f = ...  # f) a number (6 decimals)
answer_M13b_28h = ...  # h) the name of an exception, as text
answer_M13b_28b = answer_M13b_28c = answer_M13b_28d = answer_M13b_28g = ...
# Each block stands alone: compute inside it what it needs, so that "Run all" goes on when a function is not written.
with wbm.attempt("M13b.28b"):        # your lagrange_equality_qp is called here: ⏳ while it is not written
    answer_M13b_28b = ...  # b) True or False
with wbm.attempt("M13b.28c"):        # your rbf_kernel
    answer_M13b_28c = ...  # c) True or False
with wbm.attempt("M13b.28d"):        # your polynomial_kernel
    answer_M13b_28d = ...  # d) True or False
with wbm.attempt("M13b.28g"):        # your hinge_loss
    answer_M13b_28g = ...  # g) True or False

In [ ]:
for letter, answer in zip("abcdefgh", [answer_M13b_28a, answer_M13b_28b, answer_M13b_28c, answer_M13b_28d,
                                        answer_M13b_28e, answer_M13b_28f, answer_M13b_28g, answer_M13b_28h]):
    wbm.check(f"M13b.28{letter}", answer)

## Partie B · Une prédiction, une expérience et une figure

Fiche §1, §4 et §7. Le prix d'une contrainte, le pas d'une descente de sous-gradient, et le point de Lagrange sur une figure.

### Ex M13b.29 — Prédire : comment change le minimum quand on relâche la contrainte, et ce que dit λ 🔮 ★ ⏱️ 10 min
**Objectif :** prévoir, à partir du multiplicateur de Lagrange, comment le minimum réagit quand on déplace la contrainte, et comment le multiplicateur change quand on réécrit le problème, puis le vérifier.  
**Prérequis :** fiche §1 et §4 · **Parcours :** C, E

On minimise $f(x, y) = 1{,}5x^2 - xy + 2y^2 - 3x - 2y$ sous la contrainte $2x - y = c$, avec le lagrangien $f + \lambda(2x - y - c)$ (la cellule fournie l'écrit sous la forme $\frac{1}{2}\mathbf{x}^\top\mathbf{Q}\mathbf{x} + \mathbf{q} \cdot \mathbf{x}$, contrainte $\mathbf{a} \cdot \mathbf{x} = c$). Pour $c = 5$, la solution est $(x, y) = (2{,}8 ;\ 0{,}6)$, avec le multiplicateur $\lambda = -2{,}4$. Pour chaque question, choisis le numéro de la bonne réponse :
- a) quand $c$ passe de 5 à 5,1, le minimum $f^*$ : (1) monte ; (2) baisse ; (3) ne change pas ; (4) on ne peut pas le savoir sans tout recalculer ;
- b) de combien, environ : (1) 0,024 ; (2) 0,24 ; (3) 2,4 ; (4) 24 ;
- c) si l'on multiplie $f$ par 2 (même contrainte), le multiplicateur : (1) ne change pas ; (2) est multiplié par 2 ; (3) est divisé par 2 ; (4) change de signe ;
- d) si l'on écrit la même contrainte $4x - 2y = 10$ (même $f$), le multiplicateur : mêmes choix qu'en c).

**Avant d'exécuter quoi que ce soit**, écris tes quatre prédictions (le **numéro du choix**), vérifie-les, puis lance l'expérience : elle résout le problème avec `lagrange_equality_qp` de la référence pour plusieurs valeurs de $c$, trace $f^*(c)$ et sa tangente en $c = 5$.

In [ ]:
Q_29 = np.array([[3.0, -1.0], [-1.0, 4.0]])     # f(x) = ½ xᵀ Q_29 x + LIN_29 · x = 1.5x² − xy + 2y² − 3x − 2y
LIN_29 = np.array([-3.0, -2.0])
A_29, RHS_29 = np.array([2.0, -1.0]), 5.0       # the constraint a · x = c: 2x − y = 5

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
prediction_M13b_29a = ...  # a) 1, 2, 3 or 4, written BEFORE running the experiment below
prediction_M13b_29b = ...  # b) 1, 2, 3 or 4
prediction_M13b_29c = ...  # c) 1, 2, 3 or 4
prediction_M13b_29d = ...  # d) 1, 2, 3 or 4

In [ ]:
for letter, prediction in zip("abcd", [prediction_M13b_29a, prediction_M13b_29b, prediction_M13b_29c,
                                      prediction_M13b_29d]):
    wbm.check(f"M13b.29{letter}", prediction)

**L'expérience.** Elle ne s'exécute qu'une fois tes quatre prédictions écrites.

In [ ]:
if any(answer is ... or answer is None for answer in [prediction_M13b_29a, prediction_M13b_29b, prediction_M13b_29c, prediction_M13b_29d]):
    print("⏳ Écris d'abord tes quatre prédictions, puis exécute à nouveau.")
else:
    ref_29 = wbm.load_mymath("ref", alias="mymath_ref").svm


    def solve_29(Q, lin, a, c):
        """(f*, λ) of min ½ xᵀQx + lin · x under a · x = c, with the reference lagrange_equality_qp."""
        x, lam = ref_29.lagrange_equality_qp(Q, lin, [a], [c])
        return float(0.5 * x @ Q @ x + lin @ x), float(lam[0])


    f0_29, lam0_29 = solve_29(Q_29, LIN_29, A_29, RHS_29)
    f1_29 = solve_29(Q_29, LIN_29, A_29, RHS_29 + 0.1)[0]
    print(f"a) et b) f*(5) = {f0_29:.4f}, f*(5,1) = {f1_29:.4f} : le minimum change de {f1_29 - f0_29:+.4f}, "
          f"et −λ × 0,1 = {-lam0_29 * 0.1:+.4f}".replace(".", ","))
    print(f"c) λ pour f : {lam0_29:.4f} ; pour 2f : {solve_29(2 * Q_29, 2 * LIN_29, A_29, RHS_29)[1]:.4f}".replace(".", ","))
    print(f"d) λ pour 4x − 2y = 10 : {solve_29(Q_29, LIN_29, 2 * A_29, 2 * RHS_29)[1]:.4f}".replace(".", ","))
    cs_29 = np.linspace(3.0, 7.0, 81)
    fig, ax = plt.subplots(figsize=(6.5, 3.6))
    ax.plot(cs_29, [solve_29(Q_29, LIN_29, A_29, c)[0] for c in cs_29], label="f*(c), le minimum sous 2x − y = c")
    ax.plot(cs_29, f0_29 - lam0_29 * (cs_29 - RHS_29), ls="--", label="la tangente en c = 5, de pente −λ")
    ax.plot([RHS_29], [f0_29], "o", color="C3")
    ax.set_xlabel("second membre c")
    ax.set_ylabel("minimum f*(c)")
    ax.legend()
    plt.show()

**Après l'expérience**, dans ta copie : pourquoi l'écart mesuré en b) n'est-il pas exactement $-\lambda \times 0{,}1$ (regarde la courbure de $f^*(c)$) ? Écris la stationnarité pour $2f$ et pour la contrainte multipliée par 2, et retrouve c) et d) sans calcul. Que dit un multiplicateur nul ? Pourquoi le signe de $\lambda$ dépend-il de la convention $f + \lambda g$ ou $f - \lambda g$, et pas la variation de $f^*$ ?

### Ex M13b.30 — La descente de sous-gradient sur la hinge loss : pas constant contre pas décroissant 🔬 ★★ ⏱️ 20 min
**Objectif :** mesurer comment le choix du pas règle la descente de sous-gradient sur l'objectif d'une SVM linéaire : oscillations d'un pas constant, lenteur d'un petit pas, convergence d'un pas décroissant.  
**Prérequis :** fiche §7 · **Parcours :** C

150 exemples synthétiques en dimension 3, de deux classes qui se chevauchent un peu (cellule fournie), et l'objectif $J(\mathbf{w}, b)$ d'une SVM à marge souple avec $C = 0{,}5$. La cellule fournie calcule aussi son minimum $J^*$, avec la SVM de scikit-learn (`SVC(kernel="linear", C=0.5, tol=1e-10)`, un solveur du dual) et la fonction `svm_subgradient` de la référence, `ref_30`. Écris :
- `run_subgradient(X, y, C, steps, eta)` : la descente de sous-gradient qui part de $\mathbf{w} = \mathbf{0}$, $b = 0$ et, pour $t = 1, \ldots,$ `steps`, calcule `(J, g_w, g_b) = svm_subgradient(w, b, X, y, C)` (celle de la référence, `ref_30.svm_subgradient`, ou la tienne), range $J$ dans une liste, puis fait le pas $(\mathbf{w}, b) \leftarrow (\mathbf{w}, b) - \eta_t\,(\mathbf{g}_{\mathbf{w}}, g_b)$, avec $\eta_t$ = `eta(t)`. Elle rend la liste des `steps` valeurs de $J$, la première étant $J(\mathbf{0}, 0)$.

Lance-la pour les trois réglages du dictionnaire `SCHEDULES_30` (un pas constant 0,01, un pas constant 0,001, un pas décroissant $\eta_t = 0{,}01/\sqrt{t}$), avec 300 itérations, puis calcule :
- a) les trois meilleurs écarts $\min_t J_t - J^*$, dans l'ordre de `SCHEDULES_30` (une liste, 6 décimales) ;
- b) vrai ou faux : le pas décroissant finit avec un meilleur écart plus petit que celui des deux pas constants ;
- c) le nombre d'itérations où $J$ **remonte** ($J_{t+1} > J_t$) avec le pas constant 0,01 (un entier).

Une cellule trace ensuite $J_t - J^*$ en échelle logarithmique pour les trois réglages, avec le meilleur écart vu.

Dans ta copie : pourquoi le pas constant 0,01 ne descend-il plus sous un certain niveau, et pourquoi $J$ remonte-t-il si souvent (la fiche §7 dit pourquoi ce n'est pas une méthode de descente) ? Pourquoi le pas constant 0,001 est-il si lent au début ? Que garde le pas décroissant de chacun des deux ? Pourquoi garde-t-on le **meilleur** point vu, et pas le dernier ?

In [ ]:
from sklearn.svm import SVC

rng_30 = np.random.default_rng(4230)
Y_30 = np.where(rng_30.random(150) < 0.5, -1, 1)                                     # 150 labels ±1
X_30 = rng_30.normal(0.0, 1.0, size=(150, 3)) + np.outer(Y_30, [0.9, -0.7, 0.5])    # two classes that overlap a bit
C_30, STEPS_30 = 0.5, 300
SCHEDULES_30 = {"constant 0,01": lambda t: 0.01,                                     # the step η_t at iteration t
                "constant 0,001": lambda t: 0.001,
                "décroissant 0,01/√t": lambda t: 0.01 / math.sqrt(t)}
ref_30 = wbm.load_mymath("ref", alias="mymath_ref").svm
svc_30 = SVC(kernel="linear", C=C_30, tol=1e-10).fit(X_30, Y_30)
J_STAR_30 = ref_30.svm_subgradient(svc_30.coef_[0], float(svc_30.intercept_[0]), X_30, Y_30, C_30)[0]   # min of J

In [ ]:
def run_subgradient(X, y, C, steps, eta):
    """The list of the `steps` values J(w_t, b_t) of the subgradient descent that starts at w = 0, b = 0 and, for
    t = 1, ..., steps, computes (J, g_w, g_b) = svm_subgradient(w, b, X, y, C) (ref_30.svm_subgradient, or your
    own), stores J, then steps (w, b) <- (w, b) - eta(t) * (g_w, g_b)."""
    raise NotImplementedError("w = 0 et b = 0, puis à chaque t : svm_subgradient, range J, fais le pas η_t = eta(t)")


answer_M13b_30a = answer_M13b_30b = answer_M13b_30c = ...
with wbm.attempt("M13b.30a"):        # your run_subgradient is called here: ⏳ while it is not written
    runs_30 = {name: run_subgradient(X_30, Y_30, C_30, STEPS_30, eta) for name, eta in SCHEDULES_30.items()}
    answer_M13b_30a = ...  # a) a list of 3 numbers (6 decimals), in the order of SCHEDULES_30
    answer_M13b_30b = ...  # b) True or False
    answer_M13b_30c = ...  # c) an integer

In [ ]:
for letter, answer in zip("abc", [answer_M13b_30a, answer_M13b_30b, answer_M13b_30c]):
    wbm.check(f"M13b.30{letter}", answer)

with checked("M13b.30 figure"):
    fig, ax = plt.subplots(figsize=(7.5, 3.8))
    try:
        for (name, eta), color in zip(SCHEDULES_30.items(), ("C0", "C1", "C2")):
            gaps = np.asarray(run_subgradient(X_30, Y_30, C_30, STEPS_30, eta), dtype=float) - J_STAR_30
            steps = np.arange(1, len(gaps) + 1)
            ax.semilogy(steps, np.maximum(gaps, 1e-9), color=color, lw=0.7, alpha=0.6, label=f"pas {name} : J − J*")
            ax.semilogy(steps, np.maximum(np.minimum.accumulate(gaps), 1e-9), color=color, lw=2, ls="--",
                        label=f"pas {name} : meilleur écart vu")
        ax.set_xlabel("itération t")
        ax.set_ylabel("J(w_t, b_t) − J*")
        ax.legend(fontsize=7, ncol=2)
    except BaseException:
        plt.close(fig)
        raise
    plt.show()

### Ex M13b.31 — Lignes de niveau et contrainte : voir le point de Lagrange 🎨 ★ ⏱️ 20 min
**Objectif :** trouver numériquement le minimum d'une fonction sur une courbe, et voir qu'en ce point la ligne de niveau touche la courbe et que les deux gradients sont colinéaires.  
**Prérequis :** fiche §1 · **Parcours :** C, E

On minimise $f(x, y) = (x - 3)^2 + 2(y + 1)^2$, dont les lignes de niveau sont des ellipses centrées en $(3, -1)$, sous la contrainte $g(x, y) = \frac{x^2}{2} - 2 - y = 0$ : le point doit rester sur la parabole $y = \frac{x^2}{2} - 2$. La cellule fournie donne $f$, $g$ et leurs gradients. Écris `draw_lagrange(ax)`, qui, dans l'axe reçu (sans en ouvrir d'autre) :
- trouve le point de Lagrange **par ta propre méthode numérique** : par exemple, paramètre la parabole par $x$ (le point $(x, \frac{x^2}{2} - 2)$), calcule $f$ sur une grille fine de valeurs de $x$ (`np.linspace`, des dizaines de milliers de points) et garde la plus petite ;
- trace quelques lignes de niveau de $f$ (`ax.contour` sur une grille `np.meshgrid`), dont celle qui passe par ce point, la parabole, et le point ;
- dessine en ce point les deux gradients $\nabla f$ et $\nabla g$, ramenés à la même longueur (`ax.annotate` avec une flèche, ou `ax.quiver`) ;
- met `ax.set_aspect("equal")`, une légende et un titre ;
- **rend** le point trouvé, le couple `(x, y)`.

La cellule de vérification appelle ta fonction, contrôle que le point est sur la parabole (à $10^{-3}$ près) et que les deux gradients y sont colinéaires (le sinus de leur angle sous 0,01), puis affiche ta figure.

Dans ta copie : écris le système de Lagrange de ce problème, élimine $y$ et $\lambda$, et montre que $x$ vérifie $x^3 - x - 3 = 0$. Pourquoi cette équation n'a-t-elle qu'une racine réelle (regarde les variations de $x^3 - x - 3$), et que vaut $\lambda$ ? Sur ta figure, la ligne de niveau du point traverse-t-elle la parabole, ou la touche-t-elle ? Que se passerait-il en un point de la parabole où les gradients ne sont pas colinéaires ?

In [ ]:
def f_31(x, y):
    """The objective (x − 3)² + 2(y + 1)²: its level lines are ellipses centred at (3, −1)."""
    return (x - 3) ** 2 + 2 * (y + 1) ** 2


def grad_f_31(x, y):
    """The gradient of f_31 at (x, y)."""
    return np.array([2 * (x - 3), 4 * (y + 1)], dtype=float)


def g_31(x, y):
    """The constraint g = 0: the parabola y = x²/2 − 2."""
    return x ** 2 / 2 - 2 - y


def grad_g_31(x, y):
    """The gradient of g_31 at (x, y)."""
    return np.array([x, -1.0], dtype=float)

In [ ]:
def draw_lagrange(ax):
    """In the axes received (and no other): the level lines of f_31, the parabola g_31 = 0, the Lagrange point found
    numerically (the minimum of f_31 on the parabola), and the two gradients at that point. Return the point (x, y)."""
    raise NotImplementedError("une grille fine sur la parabole, le minimum de f, puis ax.contour, ax.plot, ax.annotate")

In [ ]:
with checked("M13b.31"):
    results_31 = []

    def report_31(ok, success, failure):
        results_31.append(bool(ok))
        print(("✅ " + success) if ok else ("❌ " + failure))

    fig, ax = plt.subplots(figsize=(6.5, 6.0))
    try:
        returned_31 = []
        opened_31 = drawn(lambda axis: returned_31.append(draw_lagrange(axis)), ax)
    except BaseException:
        plt.close(fig)
        raise
    report_31(not opened_31 and bool(ax.lines or ax.collections), "draw_lagrange dessine dans l'axe reçu",
              "draw_lagrange doit dessiner dans l'axe reçu, sans ouvrir de figure (ni plt.figure ni plt.subplots)")
    try:
        x_31, y_31 = (float(v) for v in returned_31[0])
        point_ok_31 = math.isfinite(x_31) and math.isfinite(y_31)
    except (TypeError, ValueError, IndexError):
        point_ok_31 = False
    if not point_ok_31:
        report_31(False, "", "draw_lagrange doit rendre le point trouvé, un couple de deux nombres (x, y)")
    else:
        gap_31 = abs(g_31(x_31, y_31))
        report_31(gap_31 <= 1e-3, f"le point ({fr(x_31, 4)} ; {fr(y_31, 4)}) est sur la contrainte",
                  f"le point ({fr(x_31, 4)} ; {fr(y_31, 4)}) n'est pas sur la parabole : |g| = {fr(gap_31, 4)} > 0,001")
        gf_31, gg_31 = grad_f_31(x_31, y_31), grad_g_31(x_31, y_31)
        sine_31 = abs(gf_31[0] * gg_31[1] - gf_31[1] * gg_31[0]) / (np.linalg.norm(gf_31) * np.linalg.norm(gg_31) or 1.0)
        report_31(sine_31 < 1e-2, f"∇f et ∇g sont colinéaires en ce point (sinus de leur angle : {sine_31:.1e})".replace(".", ","),
                  f"∇f et ∇g ne sont pas colinéaires en ce point : le sinus de leur angle vaut {fr(sine_31, 4)}, plus "
                  "que 0,01 ; cherche le minimum de f sur la courbe plus finement")
    plt.show()
    verdict("M13b.31", all(results_31), "le point de Lagrange est juste, et la figure est dans l'axe reçu.",
            "relis les lignes ❌ ci-dessus.")

## Partie C · Erreurs classiques, savoir-faire et défi

Fiche §3, §6, §7 et §8. Quatre erreurs à trouver dans un résumé de SVM, une fonction qui vérifie les conditions KKT, puis une SVM linéaire entraînée par sous-gradient jusqu'à un objectif chiffré.

### Ex M13b.32 — Chasse à l'erreur : le signe d'un multiplicateur, une largeur de bande 1/‖w‖ au lieu de 2/‖w‖, une matrice de Gram non symétrique 🐛 ★ ⏱️ 15 min
**Objectif :** repérer dans un résumé de SVM les erreurs classiques des multiplicateurs, de la marge et des noyaux, puis calculer les bonnes valeurs.  
**Prérequis :** fiche §3, §6 et §8 · **Parcours :** C

Un collègue résume une SVM linéaire et une vérification de conditions KKT (la cellule fournie) : six exemples `X_32` et leurs labels, quatre autres points `Z_32`, un séparateur $(\mathbf{w}, b)$, le poids $C$ et un paramètre $\gamma$ ; puis un candidat `P_32` pour un petit problème à une contrainte d'inégalité, avec le multiplicateur `MU_32` qui y rend la stationnarité vraie. Sept lignes portent une lettre et un commentaire qui dit ce qu'elles doivent calculer. Quatre lignes ne font pas ce que dit leur commentaire. **Sans exécuter son code**, trouve-les.

```python
X, y, w, b = X_32, Y_32, W_32, B_32
margins = y * (X @ w + b)                                            # (A) the functional margins t_i
J = 0.5 * w @ w + C_32 * np.mean(np.maximum(0, 1 - margins))         # (B) the soft-margin objective J(w, b)
band = 1 / np.linalg.norm(w)                                         # (C) the width of the band
rho = 1 / np.linalg.norm(w)                                          # (D) the margin ρ, from the boundary to a border
K = np.exp(-GAMMA_32 * np.sum((X[:, None, :] - X[None, :, :]) ** 2, axis=2))   # (E) the RBF Gram matrix of X
G = X @ Z_32.T                                                       # (F) the linear Gram matrix of X, x_i · x_j
kkt = STATIONARY_32 and G_P_32 <= 1e-12 and abs(MU_32 * G_P_32) <= 1e-12   # (G) (P_32, MU_32) is a KKT point
```

Questions :
- a) les lettres des lignes fausses, dans l'ordre alphabétique (un texte) ;
- b) la bonne valeur de `J` (4 décimales) ;
- c) la bonne valeur de `band` (4 décimales) ;
- d) la somme de tous les coefficients de la bonne matrice `G` (4 décimales) ;
- e) la bonne valeur de `kkt` (`True` ou `False`).

Pour b) à e), écris le code corrigé dans la cellule (c'est permis : seule la recherche des lignes fausses se fait sans exécuter).

In [ ]:
X_32 = np.array([[1.5, -0.5], [0.0, 2.5], [-1.0, 1.0], [2.5, 1.5], [-2.0, -1.5], [0.5, 3.0]])   # 6 examples
Y_32 = np.array([1, -1, -1, 1, -1, 1])                                                    # their labels
Z_32 = np.array([[1.0, 0.5], [-0.5, 2.0], [3.0, -1.5], [0.0, -2.5]])                      # 4 new points
W_32, B_32, C_32, GAMMA_32 = np.array([1.2, -0.5]), 0.1, 2.0, 0.5
# A candidate for min (p₁ − 3)² + (p₂ − 0.5)² under g(p) = p₁ + 2p₂ − 6 ≤ 0: the point P_32 of the line g = 0, and
# the μ that makes the stationarity ∇f + μ∇g = 0 hold there
P_32 = np.array([3.4, 1.3])
GRAD_F_32, GRAD_G_32 = 2 * (P_32 - np.array([3.0, 0.5])), np.array([1.0, 2.0])
G_P_32 = float(P_32 @ GRAD_G_32 - 6)                                                     # g(P_32)
MU_32 = float(-(GRAD_F_32 @ GRAD_G_32) / (GRAD_G_32 @ GRAD_G_32))
STATIONARY_32 = bool(np.allclose(GRAD_F_32 + MU_32 * GRAD_G_32, 0.0))

In [ ]:
answer_M13b_32a = ...  # a) letters, as text
answer_M13b_32b = ...  # b) a number (4 decimals)
answer_M13b_32c = ...  # c) a number (4 decimals)
answer_M13b_32d = ...  # d) a number (4 decimals)
answer_M13b_32e = ...  # e) True or False

In [ ]:
for letter, answer in zip("abcde", [answer_M13b_32a, answer_M13b_32b, answer_M13b_32c, answer_M13b_32d,
                                     answer_M13b_32e]):
    wbm.check(f"M13b.32{letter}", answer)

**Après la vérification**, dans ta copie : pour chaque ligne fausse, que calcule-t-elle exactement, et comment la corriger ? Laquelle donne une matrice de la mauvaise forme, et laquelle un nombre plausible sans aucun signal ? Pourquoi les lignes (C) et (D), presque identiques, ne sont-elles pas toutes les deux fausses ? Pour (G), que dit le signe de `MU_32` sur la contrainte, et quel est le vrai minimum de ce petit problème ?

### Ex M13b.33 — Vérifier une solution KKT : faisabilité, stationnarité, complémentarité 🛠️ ★ ⏱️ 15 min
**Objectif :** écrire une fonction qui vérifie les quatre conditions KKT d'une solution candidate, pour n'importe quel problème à contraintes d'inégalité.  
**Prérequis :** fiche §3 · **Parcours :** C

Un solveur, un collègue ou toi-même proposez une solution $(\mathbf{x}, \boldsymbol{\mu})$ d'un problème $\min f(\mathbf{x})$ sous $g_i(\mathbf{x}) \le 0$ : avant de la croire, on la **vérifie** (le réflexe de la fiche). Écris `check_kkt(grad_f, constraints, x, mu, tol=1e-8)`, pour un problème quelconque : `grad_f` est une fonction qui rend $\nabla f(\mathbf{x})$, `constraints` une liste de couples `(g_i, grad_g_i)` de fonctions, `x` le point et `mu` les multiplicateurs, un par contrainte. Elle rend le dictionnaire des quatre conditions de la fiche §3, chacune `True` ou `False` :
- `"feasibility"` : chaque $g_i(\mathbf{x}) \le$ `tol` ;
- `"stationarity"` : chaque coordonnée de $\nabla f(\mathbf{x}) + \sum_i \mu_i\nabla g_i(\mathbf{x})$ est, en valeur absolue, au plus `tol` ;
- `"sign"` : chaque $\mu_i \ge -$`tol` ;
- `"complementarity"` : chaque $\lvert\mu_i\,g_i(\mathbf{x})\rvert \le$ `tol`.

La cellule fournie définit un problème : $f(\mathbf{x}) = (x_1 - 1)^2 + 3(x_2 - 4)^2$, sous $g_1(\mathbf{x}) = -x_1 + 3x_2 - 3 \le 0$ et $g_2(\mathbf{x}) = x_1 - x_2 - 5 \le 0$. La cellule de vérification essaie ta fonction sur cinq candidats : un point KKT, un point hors de l'ensemble admissible, un multiplicateur négatif, une complémentarité violée et un gradient du lagrangien non nul, et dit ✅ ou ❌ pour chacun.

Dans ta copie : vérifie à la main le premier candidat, $\mathbf{x} = (3, 2)$ et $\boldsymbol{\mu} = (4, 0)$. Pourquoi ce point est-il **le** minimum, et pas seulement un point KKT (fiche §5) ? Le troisième candidat, $(7, 2)$ avec $\boldsymbol{\mu} = (0, -12)$, vérifie trois conditions sur quatre : que dit son multiplicateur négatif ? Pourquoi faut-il une tolérance, et que se passerait-il avec `tol=0` pour un point calculé en flottants ?

In [ ]:
def grad_f_33(x):
    """The gradient of f(x) = (x₁ − 1)² + 3(x₂ − 4)²."""
    return np.array([2 * (x[0] - 1), 6 * (x[1] - 4)], dtype=float)


CONSTRAINTS_33 = [                                                       # (g_i, its gradient): g_i(x) ≤ 0
    (lambda x: -x[0] + 3 * x[1] - 3, lambda x: np.array([-1.0, 3.0])),   # g₁(x) = −x₁ + 3x₂ − 3
    (lambda x: x[0] - x[1] - 5, lambda x: np.array([1.0, -1.0])),        # g₂(x) = x₁ − x₂ − 5
]

In [ ]:
def check_kkt(grad_f, constraints, x, mu, tol=1e-8):
    """The four KKT conditions at (x, mu) for min f(x) under g_i(x) <= 0: a dict {"feasibility", "stationarity",
    "sign", "complementarity"} of booleans; constraints is a list of pairs (g_i, grad_g_i), one per multiplier."""
    raise NotImplementedError("les valeurs g_i(x), le résidu ∇f + Σ μ_i ∇g_i, puis les quatre tests à tol près")

In [ ]:
with checked("M13b.33"):
    KEYS_33 = ("feasibility", "stationarity", "sign", "complementarity")
    CASES_33 = [("un point KKT", [3.0, 2.0], [4.0, 0.0]),
                ("le minimum libre, hors de l'ensemble admissible", [1.0, 4.0], [0.0, 0.0]),
                ("un multiplicateur négatif", [7.0, 2.0], [0.0, -12.0]),
                ("une complémentarité violée", [3.5, 1.5], [5.0, 0.0]),
                ("un gradient du lagrangien non nul", [3.0, 2.0], [2.0, 0.0])]

    def expected_33(x, mu):
        """The four conditions at (x, μ), computed directly."""
        x, mu = np.asarray(x, dtype=float), np.asarray(mu, dtype=float)
        g = np.array([g_i(x) for g_i, _ in CONSTRAINTS_33], dtype=float)
        rest = grad_f_33(x) + sum(m * dg(x) for m, (_, dg) in zip(mu, CONSTRAINTS_33))
        return {"feasibility": bool(np.all(g <= 1e-8)), "stationarity": bool(np.max(np.abs(rest)) <= 1e-8),
                "sign": bool(np.all(mu >= -1e-8)), "complementarity": bool(np.all(np.abs(mu * g) <= 1e-8))}

    right_33 = 0
    for name_33, x_33, mu_33 in CASES_33:
        want_33 = expected_33(x_33, mu_33)
        got_33 = check_kkt(grad_f_33, CONSTRAINTS_33, np.array(x_33), np.array(mu_33))
        if not isinstance(got_33, dict) or set(got_33) != set(KEYS_33):
            print(f"❌ {name_33} : check_kkt doit rendre un dictionnaire de clés {', '.join(KEYS_33)}, "
                  f"reçu {str(got_33)[:120]}")
            continue
        if not all(isinstance(got_33[k], (bool, np.bool_)) for k in KEYS_33):
            print(f"❌ {name_33} : chaque valeur du dictionnaire doit être True ou False")
            continue
        wrong_33 = [k for k in KEYS_33 if bool(got_33[k]) != want_33[k]]
        shown_33 = ", ".join(f"{k} {want_33[k]}" for k in KEYS_33)
        if wrong_33:
            print(f"❌ {name_33} (x = {x_33}, μ = {mu_33}) : " + " ; ".join(
                f"{k} attendu {want_33[k]}, reçu {bool(got_33[k])}" for k in wrong_33))
        else:
            right_33 += 1
            print(f"✅ {name_33} : {shown_33}")
    verdict("M13b.33", right_33 == len(CASES_33),
            "check_kkt reconnaît le point KKT et les quatre façons d'en manquer un.",
            f"{len(CASES_33) - right_33} cas faux sur {len(CASES_33)} : relis les lignes ❌ ci-dessus.")

### Ex M13b.34 — Défi : une SVM linéaire entraînée par sous-gradient, un objectif à 1 % de celui de scikit-learn au même C 🏆 ★★★ ⏱️ 30 min
**Objectif :** entraîner une SVM linéaire par descente de sous-gradient jusqu'à un objectif chiffré, en choisissant le pas, le point gardé et un changement de variable qui rend la descente facile.  
**Prérequis :** fiche §7 ; Ex M13b.30 · **Parcours :** C

Les 400 exemples `X_34` (cellule fournie) ont 5 features, de deux classes qui se chevauchent, et ils sont **loin de l'origine** (autour de $(12, -8, 6, 15, -10)$). Avec $C = 0{,}5$, la SVM de scikit-learn, `SVC(kernel="linear", C=0.5, tol=1e-10)`, atteint l'objectif $J^*$, le minimum de $J(\mathbf{w}, b) = \frac{1}{2}\lVert\mathbf{w}\rVert^2 + C\sum_i \max(0, 1 - y_i(\mathbf{w} \cdot \mathbf{x}_i + b))$.

**L'objectif** : écrire `train_linear_svm(X, y, C, subgradient)`, qui rend un couple $(\mathbf{w}, b)$ dont l'objectif $J$ est à **moins de 1 %** au-dessus de $J^*$, par descente de sous-gradient. La fonction reçue `subgradient(w, b, X, y, C)` est `svm_subgradient` (elle rend $(J, \mathbf{g}_{\mathbf{w}}, g_b)$) : ta fonction ne doit utiliser qu'elle, **au plus 20 000 fois** et en au plus 30 secondes. La vérification compte les appels, mesure le temps, et essaie ta fonction sur `X_34`, puis sur un second dataset de 300 exemples et 3 features, lui aussi loin de l'origine. scikit-learn, `scipy` (`scipy.optimize` compris), `mymath`, `svm_subgradient` appelée directement, `X_34` et `svm_dataset` sont interdits dans ta fonction.

Une descente naïve, à pas constant, échoue : essaie-la d'abord, pour voir, et cherche pourquoi (regarde $g_b$ et les coordonnées de $\mathbf{g}_{\mathbf{w}}$ quand $b$ doit valoir plusieurs dizaines). Les indices de `04_indices.md` donnent des pistes, à ouvrir une à une.

Une dernière cellule enregistre si l'objectif est atteint.

Dans ta copie : pourquoi les exemples loin de l'origine rendent-ils la descente si difficile ? Qu'as-tu changé pour réussir, et pourquoi ce changement ne modifie-t-il pas le problème (ni $J$, ni $\lVert\mathbf{w}\rVert$), alors qu'une **mise à l'échelle** des features le modifierait ? Combien d'appels ta méthode a-t-elle utilisés ?

In [ ]:
def svm_dataset(seed, n, offset, shift, spread):
    """n examples of len(offset) features, far from the origin (around offset), of two classes ±1 that overlap."""
    rng = np.random.default_rng(seed)
    y = np.where(rng.random(n) < 0.5, -1, 1)
    X = np.asarray(offset) + rng.normal(0.0, 1.0, size=(n, len(offset))) * np.asarray(spread) + np.outer(y, shift)
    return X, y


X_34, Y_34 = svm_dataset(4234, 400, [12.0, -8.0, 6.0, 15.0, -10.0], [0.6, -0.4, 0.5, 0.3, -0.5],
                         [1.0, 1.5, 0.8, 1.2, 1.0])     # 400 examples, p = 5 features
C_34, MAX_CALLS_34, MAX_SECONDS_34 = 0.5, 20000, 30.0
ref_34 = wbm.load_mymath("ref", alias="mymath_ref").svm

In [ ]:
def train_linear_svm(X, y, C, subgradient):
    """(w, b) whose objective J is less than 1 % above its minimum, by subgradient descent, calling
    subgradient(w, b, X, y, C) -> (J, g_w, g_b) at most MAX_CALLS_34 times."""
    raise NotImplementedError("pas décroissant, meilleur point vu, exemples centrés : au plus 20 000 appels")

In [ ]:
with checked("M13b.34 défi"):
    from sklearn.svm import SVC                      # the reference objective J* (forbidden in train_linear_svm)

    problems_34, ok_34, ran_34 = [], False, False
    try:
        source_34 = inspect.getsource(train_linear_svm)
    except (OSError, TypeError):
        source_34 = ""
    banned_34 = sorted(set(re.findall(r"\b(scipy|optimize|minimize|sklearn|SVC|LinearSVC|SGDClassifier|mymath|"
                                      r"mymath_ref|ref_34|svm_subgradient|X_34|Y_34|svm_dataset)\b", source_34)))
    if banned_34:
        ran_34 = True
        problems_34.append(f"train_linear_svm utilise {', '.join(banned_34)}, interdit dans ce défi")
    cases_34 = [("le dataset X_34 (n = 400, p = 5)", X_34, Y_34),
                ("un second dataset (n = 300, p = 3)",
                 *svm_dataset(4233, 300, [-10.0, 14.0, 9.0], [0.7, 0.5, -0.6], [1.0, 1.3, 0.9]))]
    for name_t, X_t, y_t in ([] if banned_34 else cases_34):
        svc_t = SVC(kernel="linear", C=C_34, tol=1e-10).fit(X_t, y_t)
        star_t = ref_34.svm_subgradient(svc_t.coef_[0], float(svc_t.intercept_[0]), X_t, y_t, C_34)[0]
        calls_t = [0]

        def counted(w, b, X, y, C=1.0, calls=calls_t):
            calls[0] += 1
            return ref_34.svm_subgradient(w, b, X, y, C)

        start_t = time.perf_counter()
        try:
            result_t = train_linear_svm(X_t.copy(), y_t.copy(), C_34, counted)
        except NotImplementedError:
            raise
        except Exception as exc:  # noqa: BLE001
            ran_34 = True
            problems_34.append(f"train_linear_svm lève {type(exc).__name__} ({exc})")
            break
        seconds_t = time.perf_counter() - start_t
        ran_34 = True
        try:
            w_t, b_t = result_t
            w_t, b_t = np.asarray(w_t, dtype=float).reshape(-1), float(b_t)
            j_t = ref_34.svm_subgradient(w_t, b_t, X_t, y_t, C_34)[0]
        except (TypeError, ValueError):
            problems_34.append("train_linear_svm doit rendre le couple (w, b) : w, un tableau de p nombres ; b, un "
                               "nombre")
            break
        ratio_t = j_t / star_t
        print(f"{name_t} : J = {j_t:.4f} pour J* = {star_t:.4f} (J/J* = {ratio_t:.5f}), {calls_t[0]} appels de "
              f"svm_subgradient, {seconds_t:.1f} s".replace(".", ","))
        if calls_t[0] > MAX_CALLS_34:
            problems_34.append(f"{name_t} : {calls_t[0]} appels, au-dessus du budget de {MAX_CALLS_34}")
        if seconds_t > MAX_SECONDS_34:
            problems_34.append(f"{name_t} : {seconds_t:.0f} s, au-dessus des {MAX_SECONDS_34:.0f} s permises")
        if not ratio_t <= 1.01:
            problems_34.append(f"{name_t} : ton objectif est {100 * (ratio_t - 1):.2f} % au-dessus de J*, plus que "
                               f"1 %".replace(".", ","))
        if problems_34:
            break
    if problems_34:
        print("❌ Ex M13b.34 : " + " ; ".join(problems_34) + ".")
    elif ran_34:
        ok_34 = True
        print("✅ Ex M13b.34 : défi réussi : un objectif à moins de 1 % de celui de scikit-learn, sur les deux datasets, "
              "dans le budget d'appels et de temps.")
with checked("M13b.34"):
    if not ran_34:                               # ⏳ while train_linear_svm is not written
        raise NotImplementedError("train_linear_svm n'est pas encore écrite")
    wbm.check("M13b.34", bool(ok_34), computed=True)

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu résoudre un problème sous contraintes d'égalité par Lagrange, et lire un multiplicateur comme un prix ?
2. Sais-tu écrire et vérifier les conditions KKT, et dire quelles contraintes sont actives ?
3. Sais-tu reconnaître un problème convexe, et dire ce que cela garantit ?
4. Sais-tu écrire le problème d'une SVM, calculer sa hinge loss et un sous-gradient, et choisir un pas de descente ?
5. Sais-tu lire le dual d'une SVM et calculer avec les noyaux polynomial et RBF ?

**La suite** : le chapitre 13 de workbookIA (classifieurs, dont les SVM linéaires et à noyau), puis M14, les maths des ensembles de modèles.